In [ ]:
# 1. サンプルデータセットをダウンロード
%cd /content
!git clone https://github.com/wal-afk/drive_sim
%cd drive_sim
!git pull
!git restore .
!git clean -fd
%cd /content/drive_sim

!pip install -U plotly==6.9

In [ ]:
import yaml

from sim.drive_simulator import CarSim
from sim.vehicle import VehicleProp
from sim.mission_base import MissionBase
from sim.goal import GoalLine, GoalCircle
from sim.drawer import SimDrawer, MissionDrawer
from sim.worlds.type_b_world import type_b_circuit
from sim.sign import Sign

with open("config/type-b.yaml", "r") as f:
    vehicle_config = yaml.safe_load(f)

prop = VehicleProp(**vehicle_config)

# ミッション5

見えている標識(標識名はoriginal)に向かって進み、最後の標識まで行ったらUターン（180度回転）して戻ってこよう。
- 標識の間をくり返し往復（ループ）できれば正解
- シミュレータ上は、最後の標識付近を通ったあとに最初の標識付近まで戻ってきたら成功とする

## 取り組み方
1. 使える命令を理解する
2. 下のセルを実行して、ロボットの限界速度や、ロボットが存在する初期位置を把握する
3. ２つ下のセル内にプログラムを書き実行して結果を見る
4. シミュレータでうまく動いたらロボットにプログラムを書きこんで動かしてみよう（講師に声をかけてね）

|使える命令|意味|指定できる値|使い方の例|
|--|--|--|--|
|move|一定速度で前に進む|v=速度[m/秒]|move(v=0.2)|
|rotate|一定速度で回転する。w>0は左回転。w<0は右回転|w=回転速度[度/秒]|rotate(w=90)|
|rotate|一定時間だけ一定速度で回転する。w>0は左回転。w<0は右回転|w=回転速度[度/秒], t=時間[秒]|rotate(w=90, t=1.0)|
|wait|直前の命令が終わるまで待つ||wait()|
|serach|標識を見つける（複数見つかった場合は、最も近いもの）||pos = search()|
|serach|特定の標識を見つける（複数見つかった場合は、最も近いもの）|name = "標識名"|pos = search(name="original")|

- pos = search() が返す値には下記が含まれる
  - pos.x: 見つけた標識の前方位置[m]　※前方が正
  - pos.y: 見つけた標識の左右位置[m]　※左側が正、右側は負
  - pos.r: 見つけた標識への距離[m]
  - pos.theta: 見つけた標識の角度[度]　※左側が正、右側は負
  - pos.name: 見つけた標識の標識名

## 注意点
- スタート時の位置はランダムに最大10cmほどずれる
- スタート時の向きはランダムに最大5度ほどずれる

In [ ]:
class Mission5Base(MissionBase):
    def __init__(self):
        super().__init__(type_b_circuit, t_max=40)
        # シミュレータでは「最後の標識付近 → 最初の標識付近」の順に通過したら成功とする
        self.goals = [
            GoalCircle((4.2, 0.7), 0.3, should_stop=False),
            GoalCircle((2.7, -0.1), 0.3, should_stop=False),
        ]
        self.initial_xy = (1.7, 0.0)
        self.random_d_xy = (0.1, 0.1)
        self.random_d_yaw_deg = 5
        self.set_signs(
            [
                Sign(x=2.7, y=-0.1, name="original"),
                Sign(x=3.5, y=0.1, name="original"),
                Sign(x=4.2, y=0.7, name="original"),
            ]
        )


print("最大速度", prop.max_velocity, "m/秒")
print("最大回転速度", prop.max_rotate_deg, "度/秒")
MissionDrawer(Mission5Base()).show()

In [ ]:
class Mission5(Mission5Base):
    @staticmethod
    def command_func(alive, *, move, rotate, wait, search, **kwargs):
        # ヒントとして、ミッション1の回答（標識に向かって進み、標識が見つからなかったら停止する）を記載済み
        # 「searchして標識が見つからなかった場合」のプログラムを書き換えよう

        while alive():
            pos = search(name="original")
            if pos is None:
                # ####### ここから下に「標識が見つからなかった場合」のプログラムを書こう
                move(v=0)
                # ####### ここより上にプログラムを書こう
            else:
                if pos.theta > 5:
                    rotate(w=45)
                elif pos.theta < -5:
                    rotate(w=-45)
                else:
                    move(v=0.2)
        # ####### プログラムを書いた後にセルを実行し結果を確認しよう


sim = CarSim(prop, Mission5())
sim.run()
SimDrawer(sim).show()

# ミッション5のヒント
- 標識に向かって進む部分は、ミッション1と同じでよい
- 最後の標識の近くまで来ると、標識がカメラに映らなくなる（search()の結果がNoneになる）
- 「標識が見つからなくなったら、その場でUターン（180度回転）する」というプログラムにしてみよう
  - 180度回転するには、rotateに回転速度と時間を指定して、waitで回転が終わるまで待とう
  - 例：90度/秒で2秒回転すると180度回転する
- Uターンした後は、また標識に向かって進めばよい